In [6]:
import requests
import pandas as pd

In [7]:
url = "https://orchestrator.pgatour.com/graphql"

In [8]:
# provided by inspecting and looking at the Network tab. Basically borrowing the API key that the 
# PGA website uses 
headers = {
    "x-api-key": "da2-gsrx5bibzbb4njvhl7t37wqyl4"
}

In [9]:
body = {
    "operationName": "StatDetails",
    "variables": {
        "tourCode": "R",
        "statId": "101",
        "year": 2022
    },
    "query": "query StatDetails($tourCode: TourCode!, $statId: String!, $year: Int) { statDetails(tourCode: $tourCode, statId: $statId, year: $year) { rows { ... on StatDetailsPlayer { playerName rank stats { statValue } } } } }"
}

response = requests.post(url, json=body, headers=headers)
print(response.status_code)

200


In [10]:
# look at the raw data that came back
print(response.json())

{'data': {'statDetails': {'rows': [{'playerName': 'Cameron Champ', 'rank': 1, 'stats': [{'statValue': '321.4'}, {'statValue': '35,996'}, {'statValue': '112'}]}, {'playerName': 'Rory McIlroy', 'rank': 2, 'stats': [{'statValue': '321.3'}, {'statValue': '38,561'}, {'statValue': '120'}]}, {'playerName': 'Cameron Young', 'rank': 3, 'stats': [{'statValue': '319.3'}, {'statValue': '52,373'}, {'statValue': '164'}]}, {'playerName': 'Wyndham Clark', 'rank': 4, 'stats': [{'statValue': '319.0'}, {'statValue': '59,335'}, {'statValue': '186'}]}, {'playerName': 'Jon Rahm', 'rank': 5, 'stats': [{'statValue': '318.9'}, {'statValue': '44,646'}, {'statValue': '140'}]}, {'playerName': 'Matthew Wolff', 'rank': 6, 'stats': [{'statValue': '318.4'}, {'statValue': '29,926'}, {'statValue': '94'}]}, {'playerName': 'Joseph Bramlett', 'rank': 7, 'stats': [{'statValue': '318.3'}, {'statValue': '51,563'}, {'statValue': '162'}]}, {'playerName': 'Brandon Hagy', 'rank': 8, 'stats': [{'statValue': '317.9'}, {'statValue'

In [21]:
#This loops through every player the API returned, skips any empty rows, and for each real player it grabs their name and their main stat value and adds it to a list. At the end we convert that list into a table.
def get_stat(stat_id, year):
    
    # define the request body
    body = {
        "operationName": "StatDetails",
        "variables": {
            "tourCode": "R",
            "statId": stat_id,
            "year": year
        },
        "query": "query StatDetails($tourCode: TourCode!, $statId: String!, $year: Int) { statDetails(tourCode: $tourCode, statId: $statId, year: $year) { rows { ... on StatDetailsPlayer { playerName rank stats { statValue } } } } }"
    }
    #spend time understanding query and why it works (and i dont have to use scrapi) rest api and pseudo rest api !!!!!!!!!!
    #and why it was so easy to get their api key !!!!!!!!!!!!
    #explicit in how i got the api key and what my steps were !!!!!!!!!!!!!
    
    # call the API
    response = requests.post(url, json=body, headers=headers)
    
    # pull out just the rows of player data
    rows = response.json()['data']['statDetails']['rows']
    
    # loop through each player and pull out name and first stat value
    players = []
    for row in rows:
        if 'playerName' in row:
            players.append({
                'player_name': row['playerName'],
                'value': row['stats'][0]['statValue']
            })
    
    # convert to a pandas dataframe
    df = pd.DataFrame(players)
    
    return df

In [18]:
# test with driving distance for 2022
test = get_stat("101", 2023)
print(test)

          player_name  value
0        Rory McIlroy  326.3
1         Peter Kuest  321.7
2    Brandon Matthews  321.3
3       Cameron Champ  317.9
4    Nicolai Hojgaard  317.7
..                ...    ...
188      Brendon Todd  282.0
189         Zac Blair  281.6
190    William McGirt  280.0
191   David Lingmerth  278.5
192      Brian Stuard  271.5

[193 rows x 2 columns]


In [16]:
# test with a different stat and year
test2 = get_stat("103", 2023)
print(test2)

           player_name   value
0    Scottie Scheffler  74.43%
1          Peter Kuest  73.79%
2         Ludvig Åberg  73.33%
3             Kevin Yu  72.61%
4         Russell Knox  71.69%
..                 ...     ...
188       Brian Stuard  62.40%
189      Lucas Herbert  62.10%
190      Danny Willett  61.19%
191    David Lingmerth  61.05%
192    Dylan Frittelli  59.62%

[193 rows x 2 columns]


In [22]:
#creating a loop for all years and stats I want 

years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

#all of the stat_ids were found in stats main page, inspect, network, click on biggest graphql, then response tab, and searching each one 
stats = {
    "scoring_avg": "120",
    "driving_dist": "101",
    "fairway_pct": "102",
    "gir": "103",
    "SG_off_the_tee": "02567",
    "SG_approach": "02568",
    "SG_around_the_green": "02569",
    "SG_putting": "02564"
}
    

In [24]:
# main loop - collect all stats for all years
all_years = []

for year in years:
    print(f"scraping {year}...")
    
    # start with an empty dataframe for this year
    year_df = None
    
    for stat_name, stat_id in stats.items():
        
        # get the data for this stat and year
        df = get_stat(stat_id, year)
        
        # rename the value column to the stat name so we know what it is
        df = df.rename(columns={"value": stat_name})
        
        # merge with the other stats for this year
        if year_df is None:
            year_df = df
        else:
            year_df = year_df.merge(df, on="player_name", how="outer")
    
    # add a year column
    year_df["year"] = year
    
    # add this year to our list
    all_years.append(year_df)

# stack all years together
final_df = pd.concat(all_years, ignore_index=True)
print("done!")
print(final_df.shape)

scraping 2019...
scraping 2020...
scraping 2021...
scraping 2022...
scraping 2023...
scraping 2024...
scraping 2025...
done!
(1327, 10)


In [25]:
print(final_df.head(10))

      player_name scoring_avg driving_dist fairway_pct    gir SG_off_the_tee  \
0  Aaron Baddeley      70.783        286.3       57.74  63.33          -.460   
1      Aaron Wise      70.739        302.6       61.75  68.60           .533   
2   Abraham Ancer      70.580        293.3       70.21  66.79           .575   
3     Adam Hadwin      70.545        291.2       67.77  67.55           .310   
4       Adam Long      71.464        292.0       66.51  65.28          -.036   
5     Adam Schenk      70.817        300.8       61.27  68.61           .080   
6      Adam Scott      69.693        299.3       59.91  68.94           .285   
7   Adam Svensson      71.042        289.8       65.24  69.40           .213   
8      Alex Cejka      72.076        278.0       67.19  64.44          -.092   
9      Alex Noren      71.046        301.1       62.38  65.28           .083   

  SG_approach SG_around_the_green SG_putting  year  
0       -.340                .460       .654  2019  
1       -.085

In [27]:
# save to csv
final_df.to_csv("pga_tour_2019_2025.csv", index=False)
print("saved!")

saved!


In [29]:
import os
print(os.path.exists("pga_tour_2019_2025.csv"))

True
